In [1]:
import numpy as np
# import matplotlib.pyplot as plt
import pandas as pd
from collections import defaultdict
import pickle
from bisect import bisect_left, bisect_right

In [2]:
import neal
import hybrid
from hybrid.core import State
from hybrid.utils import min_sample,updated_sample
np.set_printoptions(precision=5, linewidth=2000, suppress=True)
from hybrid.decomposers import EnergyImpactDecomposer
import dimod

In [3]:
#import csv file as dataframe take first column as index
dPNR = pd.read_csv('data_files/PRMI_DM_ALL_PNRs_cleaned.csv')
dCan = pd.read_csv('data_files/PRMI-DM_TARGET_FLIGHTS.csv')
dAva = pd.read_csv('data_files/PRMI-DM-AVAILABLE_FLIGHTS.csv')

In [ ]:
cans = dCan['DEP_KEY'].values
dPNR['if_can'] = dPNR['DEP_KEY'].isin(cans).astype(int)
dPNR=dPNR[['RECLOC','OPER_OD_ORIG_CD','OPER_OD_DEST_CD','DEP_KEY','ORIG_CD','DEST_CD','FLT_NUM','ARR_DTMZ','DEP_DTMZ','CVM','PAX_CNT','if_can']]
dAva=dAva[['DEP_KEY','ORIG_CD','DEST_CD','C_AVAIL_CNT','C_AUL_CNT','Y_AUL_CNT','Y_AVAIL_CNT','DEP_DTMZ','ARR_DTMZ']]
dPNR['ID'] = dPNR['RECLOC'].astype(str) + dPNR['OPER_OD_ORIG_CD']

dPNR = dPNR.sort_values(by=['RECLOC', 'DEP_DTMZ']).reset_index(drop=True)  #! dont change, important
dPNR['leg#'] = dPNR.groupby('ID').cumcount() + 1

for col in ['DEP_DTMZ','ARR_DTMZ']:
    dPNR[col] = pd.to_datetime(dPNR[col])
    dAva[col] = pd.to_datetime(dAva[col])   

dAva['AVAIL_CNT'] = np.maximum(dAva['C_AVAIL_CNT'], 0) + np.maximum(dAva['Y_AVAIL_CNT'], 0)
#drop rows in dAva with 'AVAIL_CNT'==0
dAva = dAva[dAva['AVAIL_CNT'] != 0]

In [5]:
dPNR['#legs'] = dPNR.groupby('ID')['ID'].transform('count')
dPNR['#legs_can'] = dPNR.groupby('ID')['if_can'].transform('sum')

In [6]:
data='D7'
dPNR_s = dPNR

In [7]:
print(dPNR_s.groupby(['leg#','if_can'])['ID'].count())
print(dPNR_s[dPNR_s['if_can']==1].groupby(['#legs','#legs_can'])['ID'].count())

leg#  if_can
1     0         16151
      1          9640
2     0         12490
      1          5435
3     0            59
      1            15
4     0            15
      1             4
5     0             1
6     0             1
Name: ID, dtype: int64
#legs  #legs_can
1      1            4030
2      1            9223
       2            1794
3      1              30
4      1               8
       2               8
6      1               1
Name: ID, dtype: int64


In [8]:
dCPNR_s = dPNR_s[dPNR_s['if_can']==1]

In [9]:
origs_dests = set(map(tuple, dCPNR_s[['ORIG_CD', 'DEST_CD']].values))
dAva_s = dAva[dAva[['ORIG_CD', 'DEST_CD']].apply(tuple, axis=1).isin(origs_dests)]

In [10]:
NCP_s=len(dCPNR_s)
NAF_s=len(dAva_s)
NuCP_s=len(dCPNR_s[dCPNR_s['leg#']==1])

print('Total number of PNRs -',len(dPNR_s))
print('Total number of cancelled PNRs -',NCP_s)
print('Candidate flights for these cancellations-',NAF_s)
ncp_s,nas_s=sum(dCPNR_s['PAX_CNT']),sum([x for x in dAva_s['C_AVAIL_CNT'].values if x>0])+sum([x for x in dAva_s['Y_AVAIL_CNT'].values if x>0])

print('Number of cancelled passengers-',ncp_s)
print('Number of available seats-',nas_s)



Total number of PNRs - 43811
Total number of cancelled PNRs - 15094
Candidate flights for these cancellations- 2360
Number of cancelled passengers- 23942
Number of available seats- 58441


In [11]:
# Create a new column 'NXT_FLT_DEP' initialized with NaN
# dCPNR_s['ARR_BEF'] = pd.NaT
# dCPNR_s['DEP_AFT'] = pd.NaT
# # Iterate over each unique ID
# unique_ids = dCPNR_s['ID'].unique()
# for unique_id in unique_ids:
#     id_rows = dPNR_s[dPNR_s['ID'] == unique_id]
#     if (1 in list(dCPNR_s[dCPNR_s['ID'] == unique_id]['leg#'])) and (2 in list(dCPNR_s[dCPNR_s['ID'] == unique_id]['leg#'])):
#         last_leg_arr_dtmz = id_rows[id_rows['leg#'] == 2]['ARR_DTMZ'].values[0]
#         first_leg_dep_dtmz = id_rows[id_rows['leg#'] == 1]['DEP_DTMZ'].values[0]
        
#         TD_L1=(id_rows[id_rows['leg#'] == 1]['ARR_DTMZ'].values[0]-id_rows[id_rows['leg#'] == 1]['DEP_DTMZ'].values[0])/1.2
#         TD_L2=(id_rows[id_rows['leg#'] == 2]['ARR_DTMZ'].values[0]-id_rows[id_rows['leg#'] == 2]['DEP_DTMZ'].values[0])/1.2
        
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 1), 'ARR_BEF'] = last_leg_arr_dtmz + pd.Timedelta(days=3) - TD_L2
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 1), 'DEP_AFT'] = first_leg_dep_dtmz
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 2), 'ARR_BEF'] = last_leg_arr_dtmz + pd.Timedelta(days=3)
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 2), 'DEP_AFT'] = first_leg_dep_dtmz + TD_L1
#     elif 1 in list(dCPNR_s[dCPNR_s['ID'] == unique_id]['leg#']):
#         last_leg_dep_dtmz = id_rows[id_rows['leg#'] == 2]['DEP_DTMZ'].values[0]
#         first_leg_dep_dtmz = id_rows[id_rows['leg#'] == 1]['DEP_DTMZ'].values[0]

#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 1), 'ARR_BEF'] = last_leg_dep_dtmz
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 1), 'DEP_AFT'] = first_leg_dep_dtmz
#     elif 2 in list(dCPNR_s[dCPNR_s['ID'] == unique_id]['leg#']):
#         last_leg_arr_dtmz = id_rows[id_rows['leg#'] == 2]['ARR_DTMZ'].values[0]
#         first_leg_arr_dtmz = id_rows[id_rows['leg#'] == 1]['ARR_DTMZ'].values[0]
        
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 2), 'ARR_BEF'] = last_leg_arr_dtmz + pd.Timedelta(days=3)
#         dCPNR_s.loc[(dCPNR_s['ID'] == unique_id) & (dCPNR_s['leg#'] == 2), 'DEP_AFT'] = first_leg_arr_dtmz
#!! check if it works for leg=first

In [12]:
dPNR_s.head(10)

,RECLOC,OPER_OD_ORIG_CD,OPER_OD_DEST_CD,DEP_KEY,ORIG_CD,DEST_CD,FLT_NUM,ARR_DTMZ,DEP_DTMZ,CVM,PAX_CNT,if_can,ID,leg#,#legs,#legs_can
0,0,EBG,TPH,AZ20271022EBGTPH6882,EBG,TPH,6882,2027-10-22 21:35:00,2027-10-22 19:06:00,3.321928,1,1,0EBG,1,1,1
1,1,KHL,RHP,AZ20271021KHLTPH9400,KHL,TPH,9400,2027-10-21 21:29:00,2027-10-21 14:38:00,5.689050,3,0,1KHL,1,2,1
2,1,KHL,RHP,AZ20271021TPHRHP12465,TPH,RHP,12465,2027-10-22 02:55:00,2027-10-21 23:25:00,5.689050,3,1,1KHL,2,2,1
3,1,RHP,KHL,AZ20271104RHPTPH3571,RHP,TPH,3571,2027-11-05 04:25:00,2027-11-05 00:57:00,5.689050,3,0,1RHP,1,2,0
4,1,RHP,KHL,AZ20271104TPHKHL548,TPH,KHL,548,2027-11-05 12:53:00,2027-11-05 05:51:00,5.689050,3,0,1RHP,2,2,0
5,2,CHC,OLT,AZ20271021CHCTPH2245,CHC,TPH,2245,2027-10-21 17:52:00,2027-10-21 11:07:00,5.855710,1,0,2CHC,1,2,1
6,2,CHC,OLT,AZ20271021TPHOLT17144,TPH,OLT,17144,2027-10-22 06:19:00,2027-10-21 23:07:00,5.855710,1,1,2CHC,2,2,1
7,2,OLT,CHC,AZ20271106OLTTPH10853,OLT,TPH,10853,2027-11-07 17:43:00,2027-11-07 11:19:00,5.855710,1,0,2OLT,1,2,0
8,2,OLT,CHC,AZ20271107TPHCHC5640,TPH,CHC,5640,2027-11-08 09:51:00,2027-11-08 02:51:00,5.855710,1,0,2OLT,2,2,0
9,3,ZFM,FHY,AZ20271021ZFMTPH10080,ZFM,TPH,10080,2027-10-21 17:21:00,2027-10-21 10:36:00,5.184875,3,0,3ZFM,1,2,1


In [13]:
dCPNR_s.head(10)

,RECLOC,OPER_OD_ORIG_CD,OPER_OD_DEST_CD,DEP_KEY,ORIG_CD,DEST_CD,FLT_NUM,ARR_DTMZ,DEP_DTMZ,CVM,PAX_CNT,if_can,ID,leg#,#legs,#legs_can
0,0,EBG,TPH,AZ20271022EBGTPH6882,EBG,TPH,6882,2027-10-22 21:35:00,2027-10-22 19:06:00,3.321928,1,1,0EBG,1,1,1
2,1,KHL,RHP,AZ20271021TPHRHP12465,TPH,RHP,12465,2027-10-22 02:55:00,2027-10-21 23:25:00,5.689050,3,1,1KHL,2,2,1
6,2,CHC,OLT,AZ20271021TPHOLT17144,TPH,OLT,17144,2027-10-22 06:19:00,2027-10-21 23:07:00,5.855710,1,1,2CHC,2,2,1
10,3,ZFM,FHY,AZ20271021TPHFHY5119,TPH,FHY,5119,2027-10-21 22:30:00,2027-10-21 20:34:00,5.184875,3,1,3ZFM,2,2,1
15,4,QXG,GIW,AZ20271021TPHGIW15865,TPH,GIW,15865,2027-10-22 14:33:00,2027-10-22 09:42:00,2.459678,3,1,4QXG,3,3,1
16,5,EYF,EBG,AZ20271021EYFTPH19533,EYF,TPH,19533,2027-10-21 19:21:00,2027-10-21 17:53:00,1.613300,1,1,5EYF,1,2,1
22,6,VUY,EID,AZ20271022VUYTPH5659,VUY,TPH,5659,2027-10-22 19:26:00,2027-10-22 18:02:00,2.320013,1,1,6VUY,1,2,1
24,7,PGM,CLT,AZ20271021PGMTPH18891,PGM,TPH,18891,2027-10-21 19:17:00,2027-10-21 17:10:00,2.321463,1,1,7PGM,1,2,1
29,8,TPH,QRD,AZ20271020TPHQRD1370,TPH,QRD,1370,2027-10-21 00:13:00,2027-10-20 20:47:00,0.394000,2,1,8TPH,1,1,1
30,9,EYF,TPH,AZ20271021EYFTPH7127,EYF,TPH,7127,2027-10-22 01:05:00,2027-10-21 23:37:00,0.082100,1,1,9EYF,1,1,1


In [17]:
dCPNR_s.index[2]

6

In [ ]:
dCPNR_s['ARR_BEF'] = pd.NaT
dCPNR_s['DEP_AFT'] = pd.NaT
dCPNR_s['#cons_can'] = pd.NaT
# Iterate over each unique ID
skip=0
for ind in dCPNR_s.index:
    if (dPNR_s.loc[ind]['#legs']==1)|(dPNR_s.loc[ind]['leg#']!=dPNR_s.loc[ind]['#legs'] and dPNR_s.loc[ind+1]['if_can']==0)|(dPNR_s.loc[ind]['#legs']!=1 and dPNR_s.loc[ind]['leg#']==dPNR_s.loc[ind]['#legs'] and dPNR_s.loc[ind-1]['if_can']==0):
        dCPNR_s.loc[ind, '#cons_can']=1
        if dPNR_s.loc[ind]['leg#']==1:
            dCPNR_s.loc[ind, 'DEP_AFT'] = dPNR_s.loc[ind, 'DEP_DTMZ']
        else:
            dCPNR_s.loc[ind, 'DEP_AFT'] = dPNR_s.loc[ind-1, 'ARR_DTMZ']
        if dPNR_s.loc[ind]['leg#']==dPNR_s.loc[ind]['#legs']:
            dCPNR_s.loc[ind, 'ARR_BEF'] = dPNR_s.loc[ind, 'ARR_DTMZ'] + pd.Timedelta(days=3)
        else:
            dCPNR_s.loc[ind, 'ARR_BEF'] = dPNR_s.loc[ind+1, 'DEP_DTMZ']

    elif dPNR_s.loc[ind]['leg#']!=dPNR_s.loc[ind]['#legs'] and dPNR_s.loc[ind+1]['if_can']==1:
        dCPNR_s.loc[ind, '#cons_can']=2
        TD_L1=(dPNR_s.loc[ind]['ARR_DTMZ']-dPNR_s.loc[ind]['DEP_DTMZ'])/1.2
        TD_L2=(dPNR_s.loc[ind+1]['ARR_DTMZ']-dPNR_s.loc[ind+1]['DEP_DTMZ'])/1.2

        if dPNR_s.loc[ind]['leg#']==1:
            dCPNR_s.loc[ind, 'DEP_AFT'] = dPNR_s.loc[ind, 'DEP_DTMZ']
        else:
            dCPNR_s.loc[ind, 'DEP_AFT'] = dPNR_s.loc[ind-1, 'ARR_DTMZ']

        if dPNR_s.loc[ind+1]['leg#']==dPNR_s.loc[ind]['#legs']:
            dCPNR_s.loc[ind+1, 'ARR_BEF'] = dPNR_s.loc[ind+1, 'ARR_DTMZ'] + pd.Timedelta(days=3)
        else:
            dCPNR_s.loc[ind+1, 'ARR_BEF'] = dPNR_s.loc[ind+2, 'DEP_DTMZ']

        dCPNR_s.loc[ind, 'ARR_BEF'] = dCPNR_s.loc[ind+1, 'ARR_BEF'] -TD_L2
        dCPNR_s.loc[ind+1, 'DEP_AFT'] = dCPNR_s.loc[ind, 'DEP_AFT'] + TD_L1

C:\Users\kumar\AppData\Local\Temp\ipykernel_9416\858175582.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dCPNR_s['ARR_BEF'] = pd.NaT
C:\Users\kumar\AppData\Local\Temp\ipykernel_9416\858175582.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dCPNR_s['DEP_AFT'] = pd.NaT


In [21]:
dCPNR_s = dCPNR_s.reset_index(drop=True).copy()
dAva_s = dAva_s.copy().sort_values(by=['DEP_DTMZ']).reset_index(drop=True)

In [23]:
orderedflights = dAva_s[['DEP_DTMZ','ORIG_CD','DEST_CD','AVAIL_CNT','ARR_DTMZ']].values
orderedflightsdep = orderedflights[:,0]
orderedflightsdata = orderedflights[:,1:]

In [ ]:
f_constraints=[[] for i in range(NAF_s)]
f_pax_constraints=[0]*NAF_s
ind_constraints=[[] for i in range(NCP_s)]
q_caps=[]
q_enc=[]
costs=[]
costs2=[]
# Iterate over each unique ID
skip=0
for ind in range(NCP_s):
    if dCPNR_s.loc[ind, '#cons_can']==1:
        orig, dest = dCPNR_s.loc[ind]['ORIG_CD'],dCPNR_s.loc[ind]['DEST_CD']
        dep_aft,arr_bef = dCPNR_s.loc[ind]['DEP_AFT'],dCPNR_s.loc[ind]['ARR_BEF']
        TD = (dCPNR_s.loc[ind]['ARR_DTMZ']-dCPNR_s.loc[ind]['DEP_DTMZ'])/1.2
        # f_inds=[]
        c=0
        start_index = bisect_left(orderedflightsdep, dep_aft)
        end_index = bisect_right(orderedflightsdep, arr_bef-TD)
        for f in range(start_index, end_index):
            if orderedflightsdata[f][0]==orig and orderedflightsdata[f][1]==dest:
                if orderedflightsdata[f][-2]>0 and len(f_constraints[f])<10*orderedflightsdata[f][-2] and c<10:    
                    c+=1
                                           
                    delay=orderedflightsdata[f][-1]-dCPNR_s.loc[ind]['ARR_DTMZ'] if dCPNR_s.loc[ind]['leg#']==dCPNR_s.loc[ind]['#legs'] else pd.Timedelta(days=0)
                    q_enc.append((ind,f))
                    q=len(q_enc)-1
                    q_caps.append(dCPNR_s.loc[ind]['PAX_CNT'])
                    f_constraints[f].append(q)
                    f_pax_constraints[f]+=q_caps[-1]
                    ind_constraints[ind].append(q)
                    if delay<=pd.Timedelta(days=0.25) : costs.append((q,70))
                    elif delay<=pd.Timedelta(days=0.5) : costs.append((q,50))
                    elif delay<=pd.Timedelta(days=1) : costs.append((q,40))
                    elif delay<=pd.Timedelta(days=2) : costs.append((q,30))
                    elif delay<=pd.Timedelta(days=3) : costs.append((q,20))
    elif dPNR_s.loc[ind]['leg#']!=dPNR_s.loc[ind]['#legs'] and dPNR_s.loc[ind+1]['if_can']==1:
        if skip==1:
            skip=0
        else:
            skip=1
            orig, dest = dCPNR_s.loc[ind]['ORIG_CD'],dCPNR_s.loc[ind]['DEST_CD']
            orig2, dest2 = dCPNR_s.loc[ind+1]['ORIG_CD'],dCPNR_s.loc[ind+1]['DEST_CD']
            dep_aft,arr_bef = dCPNR_s.loc[ind]['DEP_AFT'],dCPNR_s.loc[ind]['ARR_BEF']
            dep_aft2,arr_bef2 = dCPNR_s.loc[ind+1]['DEP_AFT'],dCPNR_s.loc[ind+1]['ARR_BEF']
            TD1, TD2 = (dCPNR_s.loc[ind]['ARR_DTMZ']-dCPNR_s.loc[ind]['DEP_DTMZ'])/1.2, (dCPNR_s.loc[ind+1]['ARR_DTMZ']-dCPNR_s.loc[ind+1]['DEP_DTMZ'])/1.2
            c=0
            start_index = bisect_left(orderedflightsdep, dep_aft)
            end_index = bisect_right(orderedflightsdep, arr_bef-TD1)
            start_index2 = bisect_left(orderedflightsdep, dep_aft2)
            end_index2 = bisect_right(orderedflightsdep, arr_bef2-TD2)
            # dCPNR_s.loc[ind]['ARR_BEF'] = orderedflightsdep[end_index2]
            for f2 in range(start_index2, end_index2):
                if orderedflightsdata[f2][0]==orig2 and orderedflightsdata[f2][1]==dest2:
                    if orderedflightsdata[f2][-2]>0 and len(f_constraints[f2])<10*orderedflightsdata[f2][-2]:
                        delay=orderedflightsdata[f2][-1]-dCPNR_s.loc[ind+1]['ARR_DTMZ']
                        for f in range(start_index, end_index):
                            if orderedflightsdata[f][0]==orig and orderedflightsdata[f][1]==dest:
                                if orderedflightsdata[f][-2]>0 and len(f_constraints[f])<10*orderedflightsdata[f][-2]:
                                    if orderedflightsdep[f2]>orderedflightsdata[f][-1] and c<10:
                                        c+=1
                                        try:
                                            q1=q_enc.index((ind,f))
                                        except ValueError:
                                            q_enc.append((ind,f))
                                            q1=len(q_enc)-1
                                            q_caps.append(dCPNR_s.loc[ind]['PAX_CNT'])
                                            f_pax_constraints[f]+=q_caps[-1]
                                            f_constraints[f].append(q1)
                                            ind_constraints[ind].append(len(q_enc)-1)
                                        
                                        try:
                                            q2=q_enc.index((ind+1,f2))
                                        except ValueError:
                                            q_enc.append((ind+1,f2))
                                            q2=len(q_enc)-1
                                            q_caps.append(dCPNR_s.loc[ind+1]['PAX_CNT'])
                                            f_pax_constraints[f2]+=q_caps[-1]
                                            f_constraints[f2].append(q2)
                                            ind_constraints[ind+1].append(len(q_enc)-1)
                                
                                        if delay<=pd.Timedelta(days=0.25) : costs2.append((q1,q2,35))
                                        elif delay<=pd.Timedelta(days=0.5) : costs2.append((q1,q2,25))
                                        elif delay<=pd.Timedelta(days=1) : costs2.append((q1,q2,20))
                                        elif delay<=pd.Timedelta(days=2) : costs2.append((q1,q2,15))
                                        elif delay<=pd.Timedelta(days=3) : costs2.append((q1,q2,10))
    else:                           
        

In [22]:
#computing cost matrix
# f_constraints=[[] for i in range(NAF_s)]
# ind_constraints=[[] for i in range(NCP_s)]
# q_caps=[]
# q_enc=[]
# costs=[]
# for ind in range(NCP_s//100):
#     print(ind)
#     orig, dest = dCPNR_s.loc[ind]['ORIG_CD'],dCPNR_s.loc[ind]['DEST_CD']
#     # dep_aft,arr_bef = dCPNR_s.loc[ind]['DEP_AFT'],dCPNR_s.loc[ind]['ARR_BEF']
#     # f_inds=[]
#     c=0
#     for f in range(NAF_s):
#         delay=orderedflightsdata[f][-1]-dCPNR_s.loc[ind]['ARR_DTMZ'] if dCPNR_s.loc[ind]['leg#']==dCPNR_s.loc[ind]['#legs'] else pd.Timedelta(days=0)
#         if orderedflightsdata[f][-1]<=dCPNR_s.loc[ind]['ARR_BEF'] and orderedflightsdep[f]>=dCPNR_s.loc[ind]['DEP_AFT']:
#             if orderedflightsdata[f][0]==orig and orderedflightsdata[f][1]==dest:
#                 if len(f_constraints[f])<10*orderedflightsdata[f][-2] and c<10:
#                     c+=1
#                     q_enc.append((ind,f))
#                     q_caps.append(dCPNR_s.loc[ind]['PAX_CNT'])
#                     f_constraints[f].append(len(q_enc)-1)
#                     ind_constraints[ind].append(len(q_enc)-1)
#                     if delay<=pd.Timedelta(days=0.25) : costs.append(70)
#                     elif delay<=pd.Timedelta(days=0.5) : costs.append(50)
#                     elif delay<=pd.Timedelta(days=1) : costs.append(40)
#                     elif delay<=pd.Timedelta(days=2) : costs.append(30)
#                     else : costs.append(20)

In [20]:
#assigning ancillary qubits
q_ancs=[[] for i in range(NAF_s)]
lambda2fs=[]
C=orderedflightsdata[:,-2]
for i in range(NAF_s):
    if orderedflightsdata[i][-2]>0 and f_pax_constraints[i]>orderedflightsdata[i][-2]:
        lambda2fs.append(i)
        if f_pax_constraints[i]/orderedflightsdata[i][-2]<10:
            for n in range(len(bin(orderedflightsdata[i][-2])[-1::-1])-2):
                q_enc.append((i,n))
                q_ancs[i].append(len(q_enc)-1)
N_anc=sum([len(q_ancs[i]) for i in range(NAF_s)])

In [21]:
NN=len(q_enc)
NN,N_anc

(66850, 2043)

In [23]:
from scipy.sparse import dok_matrix

#compute the cost matrix

Cost_matrix = dok_matrix((NN, NN), dtype=np.float64)

#adding costs
for (i, c) in costs:
    Cost_matrix[i, i] += -c
for (i, j, c) in costs2:
    Cost_matrix[i, j] += -c
    Cost_matrix[j, i] += -c

Constant = 0

#ind_contraints
lambda1 = 700
for q_i in ind_constraints:
    for q_i_f1 in q_i:
        for q_i_f2 in q_i:
            if q_i_f1 != q_i_f2:
                Cost_matrix[q_i_f1, q_i_f2] += lambda1

lambda2 = 100
for f in lambda2fs:
    q_f = f_constraints[f]
    if len(q_f) > 0:
        Constant += C[f]**2 * lambda2
    q1_f = q_ancs[f]
    for q_i_f in q_f:
        # first term
        for q_j_f in q_f:
            Cost_matrix[q_i_f, q_j_f] += lambda2 * q_caps[q_i_f] * q_caps[q_j_f]
        # second term
        for k in range(len(q1_f)):
            Cost_matrix[q_i_f, q1_f[k]] += lambda2 * 2**k * q_caps[q_i_f]
        # third term
        Cost_matrix[q_i_f, q_i_f] += -2 * lambda2 * C[f] * q_caps[q_i_f]

    for n in range(len(q1_f)):
        # fourth term
        for k in range(len(q1_f)):
            Cost_matrix[q1_f[n], q1_f[k]] += lambda2 * 2**(n + k)
        # fifth term
        for q_j_f in q_f:
            Cost_matrix[q1_f[n], q_j_f] += lambda2 * 2**n * q_caps[q_j_f]
        # sixth term
        Cost_matrix[q1_f[n], q1_f[n]] += -2**(n + 1) * lambda2 * C[f]


In [27]:
#check if cost matrix is symmetric
#sum obj_matrix and Cons_matrix1 and Cons_matrix2 into cost matrix
# Cost_matrix = Obj_matrix + Cons_matrix1 + Cons_matrix2


In [24]:
Q2dict={}
Q1dict={}
#!! Can optimize this
for (i, j), value in Cost_matrix.items():
    if i != j:
        Q2dict[(i, j)] = 2 * value
    else:
        Q1dict[i] = value

bqm=dimod.binary.as_bqm(Q1dict,Q2dict,Constant,'BINARY') #removed constant for now


In [25]:
init_sample=min_sample(bqm)
# for i in ind_constraints:
#     if len(i)>0:
#         init_sample[i[0]]=1
# init_sample[q_ancs[0][0]]=0
# init_sample[q_ancs[0][1]]=1
# init_sample[q_ancs[0][2]]=0
# init_sample[q_ancs[0][3]]=0
# for f in range(1,NAF_s):
#     for q in range(len(q_ancs[f])):
#         # print(C[f],bin(C[1])[-1::-1],q)
#         init_sample[q_ancs[f][q]]=int(bin(C[f])[-1::-1][q])
shots=1000
init_array=np.array(list(init_sample.values()))
init_error=bqm.energy(init_array)



In [29]:
pickle.dump(bqm, open('pickle_files/MkIII.I_s_d_'+data+'_bqm.pkl', 'wb'))

In [26]:
print(f"initial guess' energy {init_error}")

initial guess' energy 57979800.0


In [27]:
Limit='unknown'
frac=1
subiterations=1
iterations=800
# EID=EnergyImpactDecomposer(size=Limit, rolling_history=frac,rolling=True,traversal='energy')
EID=hybrid.decomposers.ComponentDecomposer(rolling=True)
iteration = (EID | hybrid.QPUSubproblemAutoEmbeddingSampler(qpu_sampler=neal.SimulatedAnnealingSampler(),sampling_params={'num_reads':shots}) | hybrid.SplatComposer())


In [27]:
init_state = State.from_samples([init_sample]*shots, bqm)
prev_state=init_state
new_state=init_state
prev_error=init_error
for iter in range(iterations):
    print('iter ',iter+1,' subQUBO size-',Limit)
    for subiter in range(subiterations):
        new_state= iteration.next(new_state)
        new_error= new_state.samples.first.energy
    if new_error<prev_error:
        print('improved!')
        prev_state=new_state
        prev_error=new_error
    else:
        new_state=prev_state
    print(new_error)

iter  1  subQUBO size- unknown
improved!
57624690.0
iter  2  subQUBO size- unknown
improved!
57293210.0
iter  3  subQUBO size- unknown
improved!
475820.0
iter  4  subQUBO size- unknown
improved!
415080.0
iter  5  subQUBO size- unknown
improved!
356870.0
iter  6  subQUBO size- unknown
improved!
349370.0
iter  7  subQUBO size- unknown
improved!
262690.0
iter  8  subQUBO size- unknown
improved!
262650.0
iter  9  subQUBO size- unknown
improved!
250850.0
iter  10  subQUBO size- unknown
improved!
196430.0
iter  11  subQUBO size- unknown
improved!
-18970.0
iter  12  subQUBO size- unknown
improved!
-47690.0
iter  13  subQUBO size- unknown
improved!
-93630.0
iter  14  subQUBO size- unknown
improved!
-93670.0
iter  15  subQUBO size- unknown
improved!
-93700.0
iter  16  subQUBO size- unknown
improved!
-93720.0
iter  17  subQUBO size- unknown
improved!
-133380.0
iter  18  subQUBO size- unknown
improved!
-163980.0
iter  19  subQUBO size- unknown
improved!
-164020.0
iter  20  subQUBO size- unknown
i

In [28]:
-NuCP_s*70 #minimum cost possible

-673750

In [29]:
ans=np.array(list(prev_state.samples.first.sample.values()))

In [37]:
#print original recloc, orig, dest, flight number, dep time, arr time, pax count, allocated flight number, new dep time, new arr time

In [38]:
ans

array([0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0], dtype=int8)

In [33]:
dAva_s_ =  dAva_s.copy()
dAva_s_.columns=['NEW_DEPKEY', 'ORIG_CD', 'DEST_CD', 'C_AVAIL_CNT', 'C_AUL_CNT','Y_AUL_CNT', 'Y_AVAIL_CNT', 'NEW_DEP_DTMZ', 'NEW_ARR_DTMZ', 'AVAIL_CNT']
C_fill=np.zeros(NAF_s)
C_empty=np.zeros(NAF_s)
PNRs_seated=[0]*NCP_s
dAPNR=pd.DataFrame(columns=['RECLOC','ORIG_CD','DEST_CD','DEP_KEY','FLT_NUM','DEP_DTMZ','ARR_DTMZ','PAX_CNT','NEW_DEPKEY','NEW_DEP_DTMZ','NEW_ARR_DTMZ'])
overbooked=0
multiplebookings=0
ans2=np.zeros(NN)
COST=0
EXTRA=0
for i in range(NN-N_anc):
    if ans[i]==1:
        # print("pnr",dCPNR_s.loc[q_enc[i][0]]['ORIG_CD']," with pax", q_caps[i], "in flight ",q_enc[i][1])
        # dCPNR_s.loc[q_enc[i][0], ['NEW_DEP_KEY', 'NEW_DEP_DTMZ', 'NEW_ARR_DTMZ']] = dAva_s.loc[q_enc[i][1], ['DEP_KEY', 'DEP_DTMZ', 'ARR_DTMZ']]
        if C_fill[q_enc[i][1]]+q_caps[i]<=C[q_enc[i][1]]:
            if PNRs_seated[q_enc[i][0]]==0:
                dAPNR = pd.concat([dAPNR, pd.DataFrame([pd.concat([dCPNR_s.loc[q_enc[i][0]][['RECLOC','ORIG_CD','DEST_CD','DEP_KEY','FLT_NUM','DEP_DTMZ','ARR_DTMZ','PAX_CNT']], dAva_s_.loc[q_enc[i][1]][['NEW_DEPKEY','NEW_DEP_DTMZ','NEW_ARR_DTMZ']]], axis=0).T])], ignore_index=True)
                C_fill[q_enc[i][1]]+=q_caps[i]
                PNRs_seated[q_enc[i][0]]=1
                ans2[i]=1
                # COST+=costs[i]
            else:
                multiplebookings+=1
                # EXTRA+=costs[i]
        else: 
            overbooked+=1
            # EXTRA+=costs[i]
for f in range(NAF_s):
    if len(q_ancs[f])>0:
        empty=sum([2**i*ans[q_ancs[f][i]] for i in range(len(q_ancs[f]))])
        # print("flight ",f," is empty with seats -",empty)
        C_empty[f]=empty

C:\Users\kumar\AppData\Local\Temp\ipykernel_26296\1897204029.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dAPNR = pd.concat([dAPNR, pd.DataFrame([pd.concat([dCPNR_s.loc[q_enc[i][0]][['RECLOC','ORIG_CD','DEST_CD','DEP_KEY','FLT_NUM','DEP_DTMZ','ARR_DTMZ','PAX_CNT']], dAva_s_.loc[q_enc[i][1]][['NEW_DEPKEY','NEW_DEP_DTMZ','NEW_ARR_DTMZ']]], axis=0).T])], ignore_index=True)


In [34]:
dAPNR['DELAY'] = dAPNR['NEW_ARR_DTMZ'] - dAPNR['ARR_DTMZ']

In [41]:
dAPNR

,RECLOC,ORIG_CD,DEST_CD,DEP_KEY,FLT_NUM,DEP_DTMZ,ARR_DTMZ,PAX_CNT,NEW_DEPKEY,NEW_DEP_DTMZ,NEW_ARR_DTMZ,DELAY
0,11606,TPH,FHY,AZ20271021TPHFHY5119,5119,2027-10-21 20:34:00,2027-10-21 22:30:00,1,AZ20271022TPHFHY5119,2027-10-22 20:34:00,2027-10-22 22:30:00,1 days 00:00:00
1,11606,CEF,TPH,AZ20271021CEFTPH15360,15360,2027-10-21 17:32:00,2027-10-21 19:27:00,1,AZ20271022CEFTPH15360,2027-10-22 17:32:00,2027-10-22 19:27:00,1 days 00:00:00
2,12584,TPH,FHY,AZ20271021TPHFHY5119,5119,2027-10-21 20:34:00,2027-10-21 22:30:00,2,AZ20271022TPHFHY5119,2027-10-22 20:34:00,2027-10-22 22:30:00,1 days 00:00:00
3,12584,CEF,TPH,AZ20271021CEFTPH15360,15360,2027-10-21 17:32:00,2027-10-21 19:27:00,2,AZ20271022CEFTPH15360,2027-10-22 17:32:00,2027-10-22 19:27:00,1 days 00:00:00
4,4078,TPH,FHY,AZ20271020TPHFHY5119,5119,2027-10-20 20:34:00,2027-10-20 22:30:00,1,AZ20271020TPHFHY13713,2027-10-20 23:00:00,2027-10-21 00:56:00,0 days 02:26:00
5,438,TPH,FHY,AZ20271021TPHFHY5119,5119,2027-10-21 20:34:00,2027-10-21 22:30:00,1,AZ20271022TPHFHY5119,2027-10-22 20:34:00,2027-10-22 22:30:00,1 days 00:00:00
6,438,CEF,TPH,AZ20271021CEFTPH15360,15360,2027-10-21 17:32:00,2027-10-21 19:27:00,1,AZ20271022CEFTPH15360,2027-10-22 17:32:00,2027-10-22 19:27:00,1 days 00:00:00
7,9628,TPH,FHY,AZ20271021TPHFHY5119,5119,2027-10-21 20:34:00,2027-10-21 22:30:00,2,AZ20271022TPHFHY5119,2027-10-22 20:34:00,2027-10-22 22:30:00,1 days 00:00:00
8,9628,CEF,TPH,AZ20271021CEFTPH15360,15360,2027-10-21 17:32:00,2027-10-21 19:27:00,2,AZ20271022CEFTPH15360,2027-10-22 17:32:00,2027-10-22 19:27:00,1 days 00:00:00


In [42]:
ans[18]

1

In [43]:
EXTRA+COST

165900

In [88]:
# mistakes=C-C_fill
# mistakes=np.where(mistakes<0,mistakes,0)
# multiplebookings=np.where(np.array(PNRs_seated)>1,1,0)
# PNRs_seated=np.where(np.array(PNRs_seated)>0,1,0)

In [35]:
print(f'number of cancelled PNRs {NCP_s}')
print(f'# of passengers cancelled {ncp_s}')
print(f'# of PNRs accomodated {sum(PNRs_seated)}')
print(f'# of passengers accomodated {sum(C_fill)}')
print(f'# of PNRs overbooked {overbooked}')
print(f'# of PNRs with multiple bookings {multiplebookings}')
print(f'# of qubits used {NN}, subqubo size {Limit}, iterations {iterations}, sampling {frac}')

number of cancelled PNRs 15047
# of passengers cancelled 23865
# of PNRs accomodated 7574
# of passengers accomodated 10749.0
# of PNRs overbooked 2
# of PNRs with multiple bookings 28
# of qubits used 66850, subqubo size unknown, iterations 800, sampling 1


In [36]:
#save dCPNR_s to csv
dAPNR.to_csv('data_files/MkIII.I_sd_sd_'+data+'.csv',index=False)


In [ ]:
pickle.dump(ans, open('pickle_files/MkIII.I_s_d_'+data+'_ans.pkl', 'wb'))

In [61]:
costs

[70, 30, 30, 20, 20]